# Kalp Hastalığı Sınıflandırması — Veri Hazırlama (Heart Disease UCI)

**Veri seti:** UCI Machine Learning Repository — Heart Disease (Cleveland alt kümesi).
Kaggle'daki *Heart Disease UCI* veri seti de bu kaynağa dayanır: 303 hasta kaydı, 14 sütun.

**Amaç:** Hastanın klinik ölçümlerinden kalp hastalığı olup olmadığını tahmin eden bir
sınıflandırma modeli kurmak. Bu notebook, modellemeden önceki hazırlık adımlarını içerir:

1. Veriyi pandas ile yükleme
2. Veriyi tanıma (sütunların anlamı, temel istatistikler)
3. Sınıf dengesi analizi (kaç "evet", kaç "hayır")
4. Veri temizleme (eksik değer kontrolü ve doldurma)
5. Kategorik değişkenlerin One-Hot Encoding ile sayısallaştırılması

---

## 0. Kütüphaneler ve ayarlar

In [1]:
import os

import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

print("pandas sürümü:", pd.__version__)
print("numpy  sürümü:", np.__version__)

pandas sürümü: 3.1.1
numpy  sürümü: 2.5.4


## 1. Veri setini yükleme

Veri önce proje içindeki yerel dosyadan (`data/processed.cleveland.data`) okunur;
dosya yoksa doğrudan UCI arşivindeki URL'den çekilir. Böylece notebook her ortamda çalışır.

Ham dosyanın **sütun başlığı yoktur**, bu yüzden sütun adlarını UCI dokümantasyonundaki
sıraya göre kendimiz veriyoruz. Eksik değerler dosyada `?` karakteriyle gösterilmiş;
`na_values=["?"]` ile bunları pandas'ın `NaN` değerine çeviriyoruz.

In [2]:
COLUMNS = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num",
]

LOCAL_PATH = "data/processed.cleveland.data"
UCI_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "heart-disease/processed.cleveland.data"
)

source = LOCAL_PATH if os.path.exists(LOCAL_PATH) else UCI_URL

df = pd.read_csv(source, header=None, names=COLUMNS, na_values=["?"])

print("Veri kaynağı :", source)
print("Satır sayısı :", df.shape[0])
print("Sütun sayısı :", df.shape[1])
df.head()

Veri kaynağı : data/processed.cleveland.data
Satır sayısı : 303
Sütun sayısı : 14


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## 2. Veriyi tanıma

### 2.1 Sütunların anlamı

| Sütun | Anlamı | Tür | Değerler |
|---|---|---|---|
| `age` | Hastanın yaşı (yıl) | Sayısal | 29 – 77 |
| `sex` | Cinsiyet | Kategorik (ikili) | 1 = erkek, 0 = kadın |
| `cp` | Göğüs ağrısı tipi (*chest pain*) | Kategorik | 1 = tipik anjina, 2 = atipik anjina, 3 = anjina dışı ağrı, 4 = asemptomatik |
| `trestbps` | Dinlenme halinde kan basıncı (mm Hg) | Sayısal | 94 – 200 |
| `chol` | Serum kolesterol (mg/dl) | Sayısal | 126 – 564 |
| `fbs` | Açlık kan şekeri > 120 mg/dl mi? | Kategorik (ikili) | 1 = evet, 0 = hayır |
| `restecg` | Dinlenme EKG sonucu | Kategorik | 0 = normal, 1 = ST-T dalga anormalliği, 2 = sol ventrikül hipertrofisi |
| `thalach` | Egzersizde ulaşılan maksimum kalp hızı | Sayısal | 71 – 202 |
| `exang` | Egzersize bağlı anjina | Kategorik (ikili) | 1 = var, 0 = yok |
| `oldpeak` | Egzersizin dinlenmeye göre oluşturduğu ST çökmesi | Sayısal | 0.0 – 6.2 |
| `slope` | Egzersiz ST segmentinin eğimi | Kategorik | 1 = yukarı eğimli, 2 = düz, 3 = aşağı eğimli |
| `ca` | Floroskopide boyanan ana damar sayısı | Sayısal (sayım, 0–3) | 0, 1, 2, 3 |
| `thal` | Talyum (talasemi) stres testi sonucu | Kategorik | 3 = normal, 6 = sabit defekt, 7 = geri dönüşlü defekt |
| `num` | **Hedef değişken** — kalp hastalığı tanısı | Kategorik (sıralı) | 0 = hastalık yok, 1–4 = hastalık var (damar daralması şiddeti) |

> **Not:** Kodlamalar orijinal UCI şemasına göre verilmiştir. Kaggle'da dolaşan bazı
> türevlerde `cp`, `restecg`, `slope` ve `thal` sıfırdan başlayacak şekilde yeniden
> numaralandırılmıştır; bu notebook ham UCI kodlamasını kullanır.

### 2.2 Veri tipleri ve temel istatistikler

In [3]:
print("=== Veri tipleri ve dolu kayıt sayıları ===")
df.info()

=== Veri tipleri ve dolu kayıt sayıları ===
<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    float64
 4   chol      303 non-null    float64
 5   fbs       303 non-null    float64
 6   restecg   303 non-null    float64
 7   thalach   303 non-null    float64
 8   exang     303 non-null    float64
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    float64
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  num       303 non-null    int64  
dtypes: float64(13), int64(1)
memory usage: 33.3 KB


In [4]:
print("=== Sayısal özet istatistikler ===")
df.describe().T

=== Sayısal özet istatistikler ===


,count,mean,std,min,25%,50%,75%,max
age,303.0,54.438944,9.038662,29.0,48.0,56.0,61.0,77.0
sex,303.0,0.679868,0.467299,0.0,0.0,1.0,1.0,1.0
cp,303.0,3.158416,0.960126,1.0,3.0,3.0,4.0,4.0
trestbps,303.0,131.689769,17.599748,94.0,120.0,130.0,140.0,200.0
chol,303.0,246.693069,51.776918,126.0,211.0,241.0,275.0,564.0
fbs,303.0,0.148515,0.356198,0.0,0.0,0.0,0.0,1.0
restecg,303.0,0.990099,0.994971,0.0,0.0,1.0,2.0,2.0
thalach,303.0,149.607261,22.875003,71.0,133.5,153.0,166.0,202.0
exang,303.0,0.326733,0.469794,0.0,0.0,0.0,1.0,1.0
oldpeak,303.0,1.039604,1.161075,0.0,0.0,0.8,1.6,6.2


In [5]:
# Kategorik olarak değerlendireceğimiz sütunların benzersiz değerleri
for kolon in ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal", "num"]:
    degerler = sorted(df[kolon].dropna().unique().tolist())
    print(f"{kolon:9s} -> {degerler}")

sex       -> [0.0, 1.0]
cp        -> [1.0, 2.0, 3.0, 4.0]
fbs       -> [0.0, 1.0]
restecg   -> [0.0, 1.0, 2.0]
exang     -> [0.0, 1.0]
slope     -> [1.0, 2.0, 3.0]
ca        -> [0.0, 1.0, 2.0, 3.0]
thal      -> [3.0, 6.0, 7.0]
num       -> [0, 1, 2, 3, 4]


## 3. Sınıf dengesi

Hedef sütun `num` 0–4 arasında değer alır ve hastalığın **şiddetini** gösterir.
Problemimiz "kalp hastalığı var mı / yok mu" şeklinde **ikili sınıflandırma** olduğu için
yeni bir `target` sütunu üretiyoruz:

- `num == 0`  →  `target = 0` (hastalık **yok** / "hayır")
- `num >= 1`  →  `target = 1` (hastalık **var** / "evet")

In [6]:
df["target"] = (df["num"] > 0).astype(int)

sayim = df["target"].value_counts().sort_index()
oran = df["target"].value_counts(normalize=True).sort_index() * 100

etiket = {0: "HAYIR (hastalık yok)", 1: "EVET  (hastalık var)"}

print("=== Hedef değişkenin sınıf dağılımı ===")
for sinif in sayim.index:
    print(f"{etiket[sinif]:24s}: {sayim[sinif]:4d} hasta  (%{oran[sinif]:.2f})")
print(f"{'TOPLAM':24s}: {sayim.sum():4d} hasta")

=== Hedef değişkenin sınıf dağılımı ===
HAYIR (hastalık yok)    :  164 hasta  (%54.13)
EVET  (hastalık var)    :  139 hasta  (%45.87)
TOPLAM                  :  303 hasta


In [7]:
# Dengesizlik oranı = çoğunluk sınıfı / azınlık sınıfı
dengesizlik = sayim.max() / sayim.min()

print(f"Çoğunluk sınıfı  : {etiket[sayim.idxmax()]} ({sayim.max()} hasta)")
print(f"Azınlık sınıfı   : {etiket[sayim.idxmin()]} ({sayim.min()} hasta)")
print(f"Dengesizlik oranı: {dengesizlik:.2f} : 1")
print()

if dengesizlik < 1.5:
    print("SONUÇ: Sınıflar DENGELİ kabul edilebilir.")
    print("  -> Dengesizlik oranı 1.5'in altında; her iki sınıf da verinin %40'ından fazlasını kapsıyor.")
    print("  -> SMOTE / class_weight gibi dengeleme tekniklerine gerek yok.")
    print("  -> Yine de accuracy tek başına yeterli değil; precision, recall, F1 ve ROC-AUC")
    print("     birlikte raporlanmalı (tıbbi teşhiste hasta olanı kaçırmamak, yani recall, kritiktir).")
elif dengesizlik < 3:
    print("SONUÇ: Sınıflar ILIMLI DENGESİZ. class_weight='balanced' denenebilir.")
else:
    print("SONUÇ: Sınıflar CİDDİ DENGESİZ. Yeniden örnekleme (SMOTE vb.) gerekir.")

Çoğunluk sınıfı  : HAYIR (hastalık yok) (164 hasta)
Azınlık sınıfı   : EVET  (hastalık var) (139 hasta)
Dengesizlik oranı: 1.18 : 1

SONUÇ: Sınıflar DENGELİ kabul edilebilir.
  -> Dengesizlik oranı 1.5'in altında; her iki sınıf da verinin %40'ından fazlasını kapsıyor.
  -> SMOTE / class_weight gibi dengeleme tekniklerine gerek yok.
  -> Yine de accuracy tek başına yeterli değil; precision, recall, F1 ve ROC-AUC
     birlikte raporlanmalı (tıbbi teşhiste hasta olanı kaçırmamak, yani recall, kritiktir).


In [8]:
# Orijinal 5 sınıflı dağılım: ikiliye indirgemeden önce nasıl görünüyordu?
siddet_etiket = {
    0: "0 - hastalık yok",
    1: "1 - hafif",
    2: "2 - orta",
    3: "3 - ileri",
    4: "4 - çok ileri",
}
print("=== Orijinal 'num' sütununun 5 sınıflı dağılımı ===")
for sinif, adet in df["num"].value_counts().sort_index().items():
    print(f"{siddet_etiket[sinif]:18s}: {adet:4d} hasta")
print()
print("Not: 1-4 sınıfları tek tek çok az örnek içerdiği için (13-55 hasta),")
print("5 sınıflı sınıflandırma yerine ikili sınıflandırma daha sağlıklı sonuç verir.")

=== Orijinal 'num' sütununun 5 sınıflı dağılımı ===
0 - hastalık yok  :  164 hasta
1 - hafif         :   55 hasta
2 - orta          :   36 hasta
3 - ileri         :   35 hasta
4 - çok ileri     :   13 hasta

Not: 1-4 sınıfları tek tek çok az örnek içerdiği için (13-55 hasta),
5 sınıflı sınıflandırma yerine ikili sınıflandırma daha sağlıklı sonuç verir.


## 4. Veri temizleme — eksik değerler

Ham dosyada eksik değerler `?` ile işaretlenmişti; yükleme sırasında `NaN`'a çevirdik.
Şimdi hangi sütunlarda ne kadar eksik olduğunu kontrol edelim.

In [9]:
eksik_sayi = df.isna().sum()
eksik_yuzde = (eksik_sayi / len(df) * 100).round(2)

eksik_tablo = pd.DataFrame({"eksik_adet": eksik_sayi, "eksik_yuzde": eksik_yuzde})
eksik_tablo = eksik_tablo[eksik_tablo["eksik_adet"] > 0].sort_values(
    "eksik_adet", ascending=False
)

print("=== Eksik değer içeren sütunlar ===")
if eksik_tablo.empty:
    print("Hiç eksik değer yok.")
else:
    print(eksik_tablo)
    print()
    print("Eksik değer içeren satır sayısı:", int(df.isna().any(axis=1).sum()),
          f"/ {len(df)}  (%{df.isna().any(axis=1).mean() * 100:.2f})")

=== Eksik değer içeren sütunlar ===
      eksik_adet  eksik_yuzde
ca             4         1.32
thal           2         0.66

Eksik değer içeren satır sayısı: 6 / 303  (%1.98)


### Hangi yöntemi seçtik ve neden?

Eksik değerler yalnızca iki sütunda ve çok az sayıda kayıtta (toplam 6 satır, verinin ~%2'si).

- **Satırları silmek** (`dropna`) mümkündü, ancak 303 satırlık küçük bir veri setinde
  6 hastayı atmak gereksiz bilgi kaybı olur.
- **Ortalama ile doldurmak** uygun değil: `ca` bir *sayım* (0–3), `thal` ise *kategorik*
  bir koddur (3/6/7). Ortalama (ör. 4.73) bu sütunlarda anlamsız, var olmayan bir değer üretir.
- **Seçilen yöntem: mod (en sık görülen değer) ile doldurma.** Her iki sütun da ayrık
  değerler aldığı için mod, var olan geçerli kategorilerden birini atar ve dağılımı bozmaz.

In [10]:
df_temiz = df.copy()

for kolon in ["ca", "thal"]:
    if df_temiz[kolon].isna().any():
        mod_deger = df_temiz[kolon].mode()[0]
        adet = int(df_temiz[kolon].isna().sum())
        df_temiz[kolon] = df_temiz[kolon].fillna(mod_deger)
        print(f"'{kolon}' sütunundaki {adet} eksik değer mod ile dolduruldu -> {mod_deger}")

print()
print("Doldurma sonrası kalan eksik değer sayısı:", int(df_temiz.isna().sum().sum()))
print("Satır sayısı korundu mu?", df_temiz.shape[0] == df.shape[0], f"({df_temiz.shape[0]} satır)")

'ca' sütunundaki 4 eksik değer mod ile dolduruldu -> 0.0
'thal' sütunundaki 2 eksik değer mod ile dolduruldu -> 3.0

Doldurma sonrası kalan eksik değer sayısı: 0
Satır sayısı korundu mu? True (303 satır)


In [11]:
# Yinelenen (tekrar eden) satır kontrolü
yinelenen = int(df_temiz.duplicated().sum())
print("Yinelenen satır sayısı:", yinelenen)
if yinelenen > 0:
    df_temiz = df_temiz.drop_duplicates().reset_index(drop=True)
    print("Yinelenen satırlar silindi. Yeni satır sayısı:", df_temiz.shape[0])
else:
    print("Temizlenecek yinelenen satır yok.")

Yinelenen satır sayısı: 0
Temizlenecek yinelenen satır yok.


## 5. Kategorik değişkenlerin sayısallaştırılması (One-Hot Encoding)

Sütunları üç gruba ayırıyoruz, çünkü her gruba farklı davranmak gerekir:

| Grup | Sütunlar | İşlem | Gerekçe |
|---|---|---|---|
| **Sayısal (sürekli)** | `age`, `trestbps`, `chol`, `thalach`, `oldpeak`, `ca` | Değişiklik yok | Değerler arasındaki büyüklük ilişkisi anlamlı |
| **İkili kategorik** | `sex`, `fbs`, `exang` | Değişiklik yok (0/1 `int`) | Zaten 0/1; One-Hot gereksiz sütun üretir |
| **Çok sınıflı kategorik** | `cp`, `restecg`, `slope`, `thal` | **One-Hot Encoding** | Kodlar arasında sıralama yok; sayı olarak bırakılırsa model yanlış sıralama öğrenir |

**Neden `cp` sayı olarak bırakılmaz?** `cp` değerleri 1, 2, 3, 4 kodlarından oluşur ama
"asemptomatik (4)" ile "tipik anjina (1)" arasında `4 > 1` gibi bir büyüklük ilişkisi yoktur.
Sayısal bırakılırsa model bunu sanki artan bir şiddet ölçeğiymiş gibi yorumlar. One-Hot
Encoding her kategori için ayrı bir 0/1 sütunu üretir ve bu yapay sıralamayı ortadan kaldırır.

`ca` sütunu ise gerçek bir **sayım** (0, 1, 2, 3 damar) olduğu için sayısal bırakıldı —
burada büyüklük ilişkisi anlamlı.

In [12]:
SAYISAL = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
IKILI = ["sex", "fbs", "exang"]
KATEGORIK = ["cp", "restecg", "slope", "thal"]

X_ham = df_temiz[SAYISAL + IKILI + KATEGORIK].copy()
y = df_temiz["target"].copy()

# İkili sütunları float -> int'e çevir (0.0/1.0 yerine 0/1)
for kolon in IKILI:
    X_ham[kolon] = X_ham[kolon].astype(int)

# Kategorik sütunları tam sayıya çevir ki sütun adları cp_1.0 değil cp_1 olsun
for kolon in KATEGORIK:
    X_ham[kolon] = X_ham[kolon].astype(int)

print("One-Hot Encoding öncesi:", X_ham.shape)
X_ham.head()

One-Hot Encoding öncesi: (303, 13)


,age,trestbps,chol,thalach,oldpeak,ca,sex,fbs,exang,cp,restecg,slope,thal
0,63.0,145.0,233.0,150.0,2.3,0.0,1,1,0,1,2,3,6
1,67.0,160.0,286.0,108.0,1.5,3.0,1,0,1,4,2,2,3
2,67.0,120.0,229.0,129.0,2.6,2.0,1,0,1,4,2,2,7
3,37.0,130.0,250.0,187.0,3.5,0.0,1,0,0,3,0,3,3
4,41.0,130.0,204.0,172.0,1.4,0.0,0,0,0,2,2,1,3


In [13]:
X = pd.get_dummies(
    X_ham,
    columns=KATEGORIK,
    prefix=KATEGORIK,
    drop_first=False,   # ağaç tabanlı modeller için tüm kategoriler kalsın
    dtype=int,          # True/False yerine 1/0
)

print("One-Hot Encoding sonrası:", X.shape)
print(f"{len(KATEGORIK)} kategorik sütun -> "
      f"{X.shape[1] - len(SAYISAL) - len(IKILI)} adet 0/1 sütununa dönüştü")
print()
print("Yeni sütunlar:")
for kolon in X.columns:
    print("  -", kolon)

One-Hot Encoding sonrası: (303, 22)
4 kategorik sütun -> 13 adet 0/1 sütununa dönüştü

Yeni sütunlar:
  - age
  - trestbps
  - chol
  - thalach
  - oldpeak
  - ca
  - sex
  - fbs
  - exang
  - cp_1
  - cp_2
  - cp_3
  - cp_4
  - restecg_0
  - restecg_1
  - restecg_2
  - slope_1
  - slope_2
  - slope_3
  - thal_3
  - thal_6
  - thal_7


In [14]:
X.head()

,age,trestbps,chol,thalach,oldpeak,ca,sex,fbs,exang,cp_1,cp_2,cp_3,cp_4,restecg_0,restecg_1,restecg_2,slope_1,slope_2,slope_3,thal_3,thal_6,thal_7
0,63.0,145.0,233.0,150.0,2.3,0.0,1,1,0,1,0,0,0,0,0,1,0,0,1,0,1,0
1,67.0,160.0,286.0,108.0,1.5,3.0,1,0,1,0,0,0,1,0,0,1,0,1,0,1,0,0
2,67.0,120.0,229.0,129.0,2.6,2.0,1,0,1,0,0,0,1,0,0,1,0,1,0,0,0,1
3,37.0,130.0,250.0,187.0,3.5,0.0,1,0,0,0,0,1,0,1,0,0,0,0,1,1,0,0
4,41.0,130.0,204.0,172.0,1.4,0.0,0,0,0,0,1,0,0,0,0,1,1,0,0,1,0,0


> **`drop_first` hakkında:** Burada `drop_first=False` kullandık; her kategori kendi
> sütununu koruyor. Bu, ağaç tabanlı modeller (Random Forest, Gradient Boosting) ve
> özellik önemi yorumlaması için daha okunabilir. Lojistik regresyon gibi doğrusal
> modellerde tam eş-doğrusallık (*dummy variable trap*) yaratmamak için
> `drop_first=True` tercih edilebilir.

### Doğrulama: veri artık modele hazır mı?

In [15]:
print("=== Doğrulama kontrolleri ===")

tumu_sayisal = all(np.issubdtype(t, np.number) for t in X.dtypes)
print(f"1) Tüm sütunlar sayısal mı?        : {tumu_sayisal}")
print(f"2) Eksik değer kaldı mı?           : {int(X.isna().sum().sum())} eksik değer")
print(f"3) Özellik matrisi (X) boyutu      : {X.shape}")
print(f"4) Hedef vektörü (y) boyutu        : {y.shape}")
print(f"5) X ve y satır sayısı uyumlu mu?  : {len(X) == len(y)}")
print(f"6) Hedef sınıf dağılımı            : {y.value_counts().sort_index().to_dict()}")
print()
print("=== Son veri tipleri ===")
print(X.dtypes.value_counts())

=== Doğrulama kontrolleri ===
1) Tüm sütunlar sayısal mı?        : True
2) Eksik değer kaldı mı?           : 0 eksik değer
3) Özellik matrisi (X) boyutu      : (303, 22)
4) Hedef vektörü (y) boyutu        : (303,)
5) X ve y satır sayısı uyumlu mu?  : True
6) Hedef sınıf dağılımı            : {0: 164, 1: 139}

=== Son veri tipleri ===
int64      16
float64     6
Name: count, dtype: int64


## 6. Hazırlanan veriyi kaydet

İşlenmiş veriyi diske yazıyoruz ki modelleme adımı (train/test ayrımı, ölçeklendirme,
model eğitimi) bu notebook'u yeniden çalıştırmak zorunda kalmadan buradan devam edebilsin.

In [16]:
os.makedirs("outputs", exist_ok=True)

islenmis = X.copy()
islenmis["target"] = y

cikti_yolu = "outputs/heart_processed.csv"
islenmis.to_csv(cikti_yolu, index=False)

print(f"Kaydedildi: {cikti_yolu}")
print(f"Boyut     : {islenmis.shape[0]} satır x {islenmis.shape[1]} sütun "
      f"({X.shape[1]} özellik + 1 hedef)")

Kaydedildi: outputs/heart_processed.csv
Boyut     : 303 satır x 23 sütun (22 özellik + 1 hedef)


## Özet

| Adım | Sonuç |
|---|---|
| **Veri yükleme** | UCI Cleveland veri seti, **303 satır × 14 sütun** |
| **Veriyi tanıma** | 14 sütunun tıbbi anlamı ve kodlaması belgelendi: 6 sürekli, 7 kategorik değişken + hedef |
| **Sınıf dengesi** | **164 "hayır"** (%54.13) / **139 "evet"** (%45.87) → dengesizlik oranı **1.18 : 1**, sınıflar **dengeli** |
| **Eksik değerler** | `ca` sütununda 4, `thal` sütununda 2 eksik (toplam 6 satır, %1.98) → **mod ile dolduruldu**, satır kaybı yok |
| **Yinelenen satır** | Yok |
| **Kategorik dönüşüm** | `cp`, `restecg`, `slope`, `thal` One-Hot Encoding ile açıldı → **13 sütun 22 sayısal özelliğe** dönüştü |
| **Çıktı** | `outputs/heart_processed.csv` (303 × 23) |

### Sonraki adımlar (bu notebook'un kapsamı dışında)

1. **Keşifsel veri analizi (EDA):** dağılım grafikleri, korelasyon ısı haritası, hedefe göre kıyaslamalar
2. **Train/test ayrımı:** `train_test_split(..., stratify=y, random_state=42)`
3. **Ölçeklendirme:** `StandardScaler` (lojistik regresyon, KNN, SVM için gerekli; ağaç modelleri için değil)
4. **Model eğitimi:** Lojistik Regresyon (temel çizgi), Random Forest, Gradient Boosting, SVM
5. **Değerlendirme:** karmaşıklık matrisi, precision / recall / F1, ROC-AUC, çapraz doğrulama
   — tıbbi teşhiste **recall** (hasta olanları kaçırmamak) özellikle önemlidir.